In [ ]:
import hashlib
import json
import shutil
from pathlib import Path

# 1. Paths Configuration
base_dir = Path(r"D:\01_University_&_Study\Samsung Innovation Campus - Immersive with MiSK Skills\CapstoneProject\MarineGuard_Datasets")
trashcan_inst_dir = base_dir / "TrashCan" / "dataset" / "dataset" / "instance_version"
output_dir = base_dir / "TrashCan_YOLO_Test"

out_img_dir = output_dir / "images"
out_lbl_dir = output_dir / "labels"
out_img_dir.mkdir(parents=True, exist_ok=True)
out_lbl_dir.mkdir(parents=True, exist_ok=True)

# 2. Merge TrashCan Train/Val Splits into an Isolated External Test Set
splits = [
    ("instances_train_trashcan.json", "train"),
    ("instances_val_trashcan.json", "val")
]

total_images = 0
nets_count = 0
ropes_count = 0
hashes = set()
duplicates = 0

for json_name, folder_name in splits:
    jf = trashcan_inst_dir / json_name
    img_src_dir = trashcan_inst_dir / folder_name
    
    data = json.loads(jf.read_text(encoding='utf-8'))
    
    # Map target category IDs (0: Nets, 1: Ropes)
    cat_map = {}
    for c in data["categories"]:
        name = c["name"].lower()
        if "net" in name:
            cat_map[c["id"]] = 0
        elif "rope" in name:
            cat_map[c["id"]] = 1
            
    # Group annotations by image ID
    img_annotations = {}
    for ann in data["annotations"]:
        cid = ann["category_id"]
        if cid in cat_map:
            img_annotations.setdefault(ann["image_id"], []).append((cat_map[cid], ann["bbox"]))
            
    # Convert coordinates to YOLO format and export images/labels
    for img_info in data["images"]:
        img_id = img_info["id"]
        if img_id not in img_annotations:
            continue
            
        file_name = img_info["file_name"]
        src_img_path = img_src_dir / file_name
        if not src_img_path.exists():
            continue
            
        # Deduplicate using MD5 file hash
        file_hash = hashlib.md5(src_img_path.read_bytes()).hexdigest()
        if file_hash in hashes:
            duplicates += 1
            continue
        hashes.add(file_hash)
        
        w_img, h_img = img_info["width"], img_info["height"]
        yolo_lines = []
        
        for cls_id, bbox in img_annotations[img_id]:
            x_min, y_min, box_w, box_h = bbox
            x_center = (x_min + box_w / 2.0) / w_img
            y_center = (y_min + box_h / 2.0) / h_img
            w_norm = box_w / w_img
            h_norm = box_h / h_img
            
            # Clamp normalized coordinates within [0.0, 1.0]
            x_center = max(0.0, min(1.0, x_center))
            y_center = max(0.0, min(1.0, y_center))
            w_norm = max(0.0, min(1.0, w_norm))
            h_norm = max(0.0, min(1.0, h_norm))
            
            yolo_lines.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")
            if cls_id == 0:
                nets_count += 1
            elif cls_id == 1:
                ropes_count += 1
                
        # Copy image and write YOLO label file
        new_name = f"trashcan_{folder_name}_{file_name}"
        shutil.copy(src_img_path, out_img_dir / new_name)
        (out_lbl_dir / f"{Path(new_name).stem}.txt").write_text("\n".join(yolo_lines), encoding='utf-8')
        total_images += 1

print("=== External Test Split: TrashCan 1.0 (YOLO Ready) ===")
print(f"  Output Folder      : {output_dir.name}")
print(f"  Total Images       : {total_images}")
print(f"  Total Label Files  : {total_images}")
print(f"  Nets (Class 0)     : {nets_count}")
print(f"  Ropes (Class 1)    : {ropes_count}")
print(f"  Background Images  : 0")
print(f"  Duplicate Images   : {duplicates}")
print("=" * 53)